# 03 - Model Training (Random Forest Only - FAST VERSION)

This notebook trains TWO Random Forest models:

1. **Classification model**: predicts the probability of high surprise-billing risk  
2. **Regression model**: predicts estimated financial exposure  

**Much faster than the full version** - focused on what matters for your presentation.

In [ ]:

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

pd.set_option("display.max_columns", 120)
os.makedirs("artifacts", exist_ok=True)

print("Loading data...")
df = pd.read_parquet("data/model_features.parquet")
print(f"Data shape: {df.shape}")
print(df.head())

In [ ]:

# Features and targets
y_class = df["proxy_surprise_bill_label"].astype(int)
y_reg = df["proxy_exposure_amount"].astype(float)

numeric_features = [
    "average_covered_charges", "average_total_payments", "Billed amount", "Medicare payment",
    "charge_ratio_inpatient", "payment_gap_inpatient", "charge_ratio_outpatient", "payment_gap_outpatient",
    "blended_charge_ratio", "blended_payment_gap", "log_avg_covered", "log_avg_payments",
    "log_billed_amount", "log_medicare_payment", "log_blended_gap",
    "is_er", "is_imaging", "is_outpatient", "is_surgery", "high_complexity_drg",
    "provider_avg_gap", "provider_gap_std", "provider_avg_ratio", "provider_record_count",
    "state_avg_gap", "state_avg_ratio", "state_median_payment", "state_record_count",
    "provider_risk_index", "state_risk_index", "service_risk_weight",
]

categorical_features = ["provider_state", "service_type"]
text_feature = "drg_text"

X = df[numeric_features + categorical_features + [text_feature]].copy()

print(f"Features shape: {X.shape}")
print(f"Classification target shape: {y_class.shape}")
print(f"Regression target shape: {y_reg.shape}")

In [ ]:

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

_, _, yreg_train, yreg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Preprocessing pipeline
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("txt", TfidfVectorizer(max_features=100, ngram_range=(1, 2)), text_feature),
    ],
    remainder="drop"
)

print("Preprocessor created")

In [ ]:

# CLASSIFICATION MODEL - Random Forest
print("\n" + "="*60)
print("TRAINING CLASSIFICATION MODEL (Random Forest)")
print("="*60)

start = time.time()

clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,      # Number of trees
        max_depth=10,          # Max depth of each tree
        min_samples_leaf=4,    # Min samples to form a leaf
        random_state=42,
        n_jobs=-1,             # Use all CPU cores
        class_weight="balanced_subsample"  # Handle class imbalance
    ))
])

print("Training...")
clf.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Training complete in {elapsed:.1f} seconds")

# Evaluate
print("\nEvaluating on test set...")
test_proba = clf.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, test_proba)
print(f"\n{'='*60}")
print(f"Classification Model Performance:")
print(f"{'='*60}")
print(f"ROC-AUC Score: {auc:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print(f"\nClassification Report:")
print(classification_report(y_test, test_pred))

In [ ]:

# REGRESSION MODEL - Random Forest
print("\n" + "="*60)
print("TRAINING REGRESSION MODEL (Random Forest)")
print("="*60)

start = time.time()

reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training...")
reg.fit(X_train, yreg_train)

elapsed = time.time() - start
print(f"Training complete in {elapsed:.1f} seconds")

# Evaluate
print("\nEvaluating on test set...")
reg_pred = reg.predict(X_test)

mae = mean_absolute_error(yreg_test, reg_pred)
rmse = mean_squared_error(yreg_test, reg_pred, squared=False)
r2 = r2_score(yreg_test, reg_pred)

print(f"\n{'='*60}")
print(f"Regression Model Performance:")
print(f"{'='*60}")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

# Additional stats
print(f"\nActual exposure range: ${yreg_test.min():,.0f} - ${yreg_test.max():,.0f}")
print(f"Predicted exposure range: ${reg_pred.min():,.0f} - ${reg_pred.max():,.0f}")

In [ ]:

# FEATURE IMPORTANCE
print("\n" + "="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

fitted_pre = clf.named_steps["preprocessor"]
fitted_model = clf.named_steps["model"]

feature_names = fitted_pre.get_feature_names_out()
importances = fitted_model.feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nTop 20 Most Important Features:")
print(fi.head(20).to_string(index=False))

In [ ]:

# VISUALIZATIONS
print("\nCreating visualizations...")

# Feature importance plot
fig, ax = plt.subplots(figsize=(10, 6))
top_fi = fi.head(15).sort_values("importance")
ax.barh(top_fi["feature"], top_fi["importance"], color="steelblue")
ax.set_xlabel("Importance")
ax.set_title("Top 15 Feature Importances - Risk Classifier")
plt.tight_layout()
plt.savefig("artifacts/feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifacts/feature_importance.png")

# Risk score distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(test_proba, bins=30, color="steelblue", alpha=0.7, edgecolor='black')
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision threshold')
ax.set_xlabel("Predicted Risk Probability")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Predicted Risk Scores")
ax.legend()
plt.tight_layout()
plt.savefig("artifacts/risk_score_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifacts/risk_score_distribution.png")

# Exposure prediction scatter
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(yreg_test, reg_pred, alpha=0.5, s=20)
ax.plot([yreg_test.min(), yreg_test.max()], [yreg_test.min(), yreg_test.max()], 'r--', lw=2)
ax.set_xlabel("Actual Exposure ($)")
ax.set_ylabel("Predicted Exposure ($)")
ax.set_title(f"Regression Model Predictions (R² = {r2:.3f})")
plt.tight_layout()
plt.savefig("artifacts/regression_predictions.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: artifacts/regression_predictions.png")

In [ ]:

# SAVE ARTIFACTS
print("\n" + "="*60)
print("SAVING MODELS AND METRICS")
print("="*60)

joblib.dump(clf, "artifacts/risk_classifier.joblib")
print("✓ Saved: artifacts/risk_classifier.joblib")

joblib.dump(reg, "artifacts/exposure_regressor.joblib")
print("✓ Saved: artifacts/exposure_regressor.joblib")

fi.to_csv("artifacts/feature_importance.csv", index=False)
print("✓ Saved: artifacts/feature_importance.csv")

# Save metrics
metrics = {
    "model_type": "Random Forest Only (Fast Version)",
    "classification_auc": float(auc),
    "regression_mae": float(mae),
    "regression_rmse": float(rmse),
    "regression_r2": float(r2),
    "training_time_seconds": "~2-5 minutes total",
}

with open("artifacts/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("✓ Saved: artifacts/model_metrics.json")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Classification AUC: {auc:.4f}")
print(f"Regression MAE: ${mae:,.2f}")
print(f"Regression RMSE: ${rmse:,.2f}")
print(f"Regression R²: {r2:.4f}")
print("\nAll artifacts saved to artifacts/ folder")
print("Ready for prediction and dashboard!")